In [ ]:
from pathlib import Path
import pandas as pd
import os

PROJECT_ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
RAW_DIR = PROJECT_ROOT / "01_raw" / "seasonality"

print("=" * 80)
print("STEP 23.1 — MBIE ADP RAW FILE INVENTORY")
print("=" * 80)

print("\nRaw folder:")
print(RAW_DIR)

files = [p for p in RAW_DIR.iterdir() if p.is_file()]

print(f"\nNumber of files found: {len(files)}")

for i, f in enumerate(files, 1):
    print(f"\nFILE {i}")
    print("Name      :", f.name)
    print("Suffix    :", f.suffix)
    print("Size MB   :", round(f.stat().st_size / (1024**2), 3))

print("\n" + "=" * 80)

if len(files) == 1:
    ADP_FILE = files[0]
    print("Selected ADP raw file:")
    print(ADP_FILE)
else:
    ADP_FILE = None
    print("STOP: Expected exactly one raw ADP file.")
    print("Please inspect the folder before continuing.")

In [ ]:
# ============================================================
# STEP 23.2 — INSPECT ADP CSV STRUCTURE
# Read header + small sample only
# ============================================================

print("=" * 80)
print("STEP 23.2 — MBIE ADP RAW SCHEMA INSPECTION")
print("=" * 80)

# Read only first 10 rows first — do NOT load the full file yet
sample = pd.read_csv(ADP_FILE, nrows=10, low_memory=False)

print("\nSample shape:")
print(sample.shape)

print("\nNumber of columns:")
print(len(sample.columns))

print("\nColumn names:")
for i, col in enumerate(sample.columns, 1):
    print(f"{i:02d}. {repr(col)}")

print("\nFirst 10 rows:")
display(sample)

print("\nData types inferred from sample:")
print(sample.dtypes)

print("\nMissing values in first 10 rows:")
print(sample.isna().sum())

print("\n" + "=" * 80)
print("STEP 23.2 COMPLETE — SAMPLE ONLY")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.3A — AUDIT NON-NUMERIC VALUES IN ADP "Value"
# ============================================================

print("=" * 80)
print("STEP 23.3A — AUDIT ADP VALUE FIELD")
print("=" * 80)

print("\nRaw dtype of Value:")
print(adp["Value"].dtype)

# Convert to numeric ONLY for QC.
# Original Value column remains unchanged.
adp["Value_numeric"] = pd.to_numeric(adp["Value"], errors="coerce")

# Values that existed in raw data but could not be converted to numbers
non_numeric_mask = (
    adp["Value"].notna() &
    adp["Value_numeric"].isna()
)

print("\nTotal rows:", f"{len(adp):,}")
print("Raw Value missing:", f"{adp['Value'].isna().sum():,}")
print("Successfully converted to numeric:",
      f"{adp['Value_numeric'].notna().sum():,}")
print("Non-numeric Value rows:",
      f"{non_numeric_mask.sum():,}")

print("\nUnique non-numeric raw values:")
print(
    adp.loc[non_numeric_mask, "Value"]
       .astype(str)
       .value_counts(dropna=False)
       .head(50)
)

print("\nExamples of rows containing non-numeric Value:")
display(
    adp.loc[
        non_numeric_mask,
        ["Month", "Area type", "Area",
         "Property", "Measure", "Value", "Flag"]
    ].head(30)
)

print("\nFlag values associated with non-numeric Value:")
print(
    adp.loc[non_numeric_mask, "Flag"]
       .value_counts(dropna=False)
       .head(30)
)

print("\n" + "=" * 80)
print("STEP 23.3A COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.3B — ADP DIMENSION + COVERAGE INVENTORY
# ============================================================

print("=" * 80)
print("STEP 23.3B — MBIE ADP DIMENSION AND COVERAGE INVENTORY")
print("=" * 80)

# ------------------------------------------------------------
# 1. Parse date explicitly
# ------------------------------------------------------------
adp["Month_parsed"] = pd.to_datetime(
    adp["Month"],
    format="%d/%m/%Y",
    errors="coerce"
)

print("\n1. BASIC STRUCTURE")
print("-" * 60)
print("Rows:", f"{len(adp):,}")
print("Columns in raw file:", 7)
print("Date parse failures:", f"{adp['Month_parsed'].isna().sum():,}")
print("Earliest month:", adp["Month_parsed"].min())
print("Latest month  :", adp["Month_parsed"].max())
print("Unique months:", adp["Month_parsed"].nunique())


# ------------------------------------------------------------
# 2. Area types
# ------------------------------------------------------------
print("\n2. AREA TYPES")
print("-" * 60)

area_summary = (
    adp.groupby("Area type", dropna=False)
       .agg(
           rows=("Value", "size"),
           unique_areas=("Area", "nunique")
       )
       .reset_index()
       .sort_values("rows", ascending=False)
)

display(area_summary)


# ------------------------------------------------------------
# 3. Exact area names by area type
# ------------------------------------------------------------
print("\n3. AREA NAMES BY AREA TYPE")
print("-" * 60)

for area_type in sorted(adp["Area type"].dropna().unique()):
    areas = sorted(
        adp.loc[adp["Area type"] == area_type, "Area"]
           .dropna()
           .unique()
    )

    print(f"\n{area_type}: {len(areas)} unique areas")
    print(areas)


# ------------------------------------------------------------
# 4. Property dimension
# ------------------------------------------------------------
print("\n4. PROPERTY VALUES")
print("-" * 60)

property_summary = (
    adp.groupby("Property", dropna=False)
       .size()
       .reset_index(name="rows")
       .sort_values(["rows", "Property"], ascending=[False, True])
)

print("Unique Property values:",
      adp["Property"].nunique(dropna=False))

display(property_summary)


# ------------------------------------------------------------
# 5. Measure dimension
# ------------------------------------------------------------
print("\n5. MEASURE VALUES")
print("-" * 60)

measure_summary = (
    adp.groupby("Measure", dropna=False)
       .agg(
           rows=("Value", "size"),
           numeric_rows=("Value_numeric", lambda x: x.notna().sum())
       )
       .reset_index()
)

measure_summary["non_numeric_rows"] = (
    measure_summary["rows"] -
    measure_summary["numeric_rows"]
)

measure_summary = measure_summary.sort_values(
    ["rows", "Measure"],
    ascending=[False, True]
)

print("Unique Measure values:",
      adp["Measure"].nunique(dropna=False))

display(measure_summary)


# ------------------------------------------------------------
# 6. Duplicate analytical keys
# ------------------------------------------------------------
print("\n6. DUPLICATE KEY CHECK")
print("-" * 60)

key_cols = [
    "Month",
    "Area type",
    "Area",
    "Property",
    "Measure"
]

duplicate_rows = adp.duplicated(
    subset=key_cols,
    keep=False
).sum()

print(
    "Rows involved in duplicate "
    "Month × Area type × Area × Property × Measure keys:",
    f"{duplicate_rows:,}"
)


# ------------------------------------------------------------
# 7. Historical candidate window
# ------------------------------------------------------------
print("\n7. PRE-STUDY HISTORICAL WINDOW: 2022-01 TO 2024-12")
print("-" * 60)

hist = adp[
    adp["Month_parsed"].between(
        pd.Timestamp("2022-01-01"),
        pd.Timestamp("2024-12-01")
    )
].copy()

print("Rows:", f"{len(hist):,}")
print("Unique months:", hist["Month_parsed"].nunique())
print("Earliest:", hist["Month_parsed"].min())
print("Latest  :", hist["Month_parsed"].max())

expected_months = pd.date_range(
    "2022-01-01",
    "2024-12-01",
    freq="MS"
)

actual_months = pd.DatetimeIndex(
    sorted(hist["Month_parsed"].dropna().unique())
)

missing_months = expected_months.difference(actual_months)

print("Expected months:", len(expected_months))
print("Missing calendar months:", len(missing_months))

if len(missing_months) > 0:
    print(list(missing_months))


# ------------------------------------------------------------
# 8. TA-specific inventory
# ------------------------------------------------------------
print("\n8. TA-LEVEL INVENTORY")
print("-" * 60)

ta_mask = (
    adp["Area type"]
    .astype(str)
    .str.strip()
    .str.casefold()
    .eq("territorial authority")
)

ta = adp.loc[ta_mask].copy()

print("TA-level rows:", f"{len(ta):,}")
print("Unique TA area names:", ta["Area"].nunique())
print("TA earliest month:", ta["Month_parsed"].min())
print("TA latest month  :", ta["Month_parsed"].max())

if len(ta) == 0:
    print("\nWARNING:")
    print("No exact Area type == 'Territorial Authority' was found.")
    print("Use the AREA TYPES table above to identify MBIE's actual label.")


print("\n" + "=" * 80)
print("STEP 23.3B COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.4 — TA-LEVEL SEASONAL MEASURE COVERAGE AUDIT
# Historical window: Jan 2022 – Dec 2024
# ============================================================

print("=" * 80)
print("STEP 23.4 — TA-LEVEL SEASONAL MEASURE COVERAGE AUDIT")
print("=" * 80)

# ------------------------------------------------------------
# 1. Correct TA filter
# ------------------------------------------------------------
ta_hist = adp[
    (adp["Area type"] == "TA") &
    (adp["Month_parsed"] >= pd.Timestamp("2022-01-01")) &
    (adp["Month_parsed"] <= pd.Timestamp("2024-12-01")) &
    (adp["Property"] == "Total") &
    (adp["Area"] != "Total New Zealand")
].copy()

print("\n1. TA HISTORICAL BASE")
print("-" * 60)
print("Rows:", f"{len(ta_hist):,}")
print("Unique TAs:", ta_hist["Area"].nunique())
print("Unique months:", ta_hist["Month_parsed"].nunique())
print("Earliest:", ta_hist["Month_parsed"].min())
print("Latest  :", ta_hist["Month_parsed"].max())


# ------------------------------------------------------------
# 2. Candidate measures
# ------------------------------------------------------------
candidate_measures = [
    "Total guest nights",
    "Occupancy rate",
    "Capacity utilisation rate"
]

cand = ta_hist[
    ta_hist["Measure"].isin(candidate_measures)
].copy()

print("\n2. CANDIDATE MEASURE COVERAGE")
print("-" * 60)

coverage = (
    cand.groupby("Measure")
        .agg(
            rows=("Value", "size"),
            numeric_rows=("Value_numeric", lambda x: x.notna().sum()),
            unique_TAs=("Area", "nunique"),
            unique_months=("Month_parsed", "nunique")
        )
        .reset_index()
)

coverage["non_numeric_rows"] = (
    coverage["rows"] - coverage["numeric_rows"]
)

coverage["numeric_pct"] = (
    100 * coverage["numeric_rows"] / coverage["rows"]
)

display(coverage)


# ------------------------------------------------------------
# 3. Non-numeric status by measure
# ------------------------------------------------------------
print("\n3. NON-NUMERIC STATUS BY MEASURE")
print("-" * 60)

for measure in candidate_measures:
    x = cand[cand["Measure"] == measure]

    print(f"\n--- {measure} ---")
    print(
        x.loc[x["Value_numeric"].isna(), "Value"]
         .value_counts(dropna=False)
    )


# ------------------------------------------------------------
# 4. Numeric month coverage by TA
# ------------------------------------------------------------
print("\n4. NUMERIC MONTH COVERAGE PER TA")
print("-" * 60)

ta_coverage = (
    cand.groupby(["Measure", "Area"])
        .agg(
            expected_rows=("Month_parsed", "size"),
            numeric_months=("Value_numeric", lambda x: x.notna().sum())
        )
        .reset_index()
)

ta_coverage["missing_or_suppressed_months"] = (
    ta_coverage["expected_rows"] -
    ta_coverage["numeric_months"]
)

for measure in candidate_measures:
    x = ta_coverage[
        ta_coverage["Measure"] == measure
    ].copy()

    print(f"\n--- {measure} ---")
    print("TAs represented:", len(x))
    print("TAs with complete 36/36 numeric months:",
          (x["numeric_months"] == 36).sum())
    print("TAs with at least 30 numeric months:",
          (x["numeric_months"] >= 30).sum())
    print("TAs with at least 24 numeric months:",
          (x["numeric_months"] >= 24).sum())
    print("TAs with zero numeric months:",
          (x["numeric_months"] == 0).sum())

    print("\nWorst coverage:")
    display(
        x.sort_values(
            ["numeric_months", "Area"]
        ).head(15)
    )


# ------------------------------------------------------------
# 5. Coverage by calendar month-of-year
# ------------------------------------------------------------
print("\n5. NUMERIC COVERAGE BY CALENDAR MONTH")
print("-" * 60)

cand["calendar_month"] = cand["Month_parsed"].dt.month

month_coverage = (
    cand.groupby(["Measure", "calendar_month"])
        .agg(
            rows=("Value", "size"),
            numeric_rows=("Value_numeric", lambda x: x.notna().sum())
        )
        .reset_index()
)

month_coverage["numeric_pct"] = (
    100 *
    month_coverage["numeric_rows"] /
    month_coverage["rows"]
)

display(month_coverage)


# ------------------------------------------------------------
# 6. Flag audit
# ------------------------------------------------------------
print("\n6. FLAG DISTRIBUTION")
print("-" * 60)

flag_summary = (
    cand.groupby(["Measure", "Flag"], dropna=False)
        .size()
        .reset_index(name="rows")
)

display(flag_summary)


print("\n" + "=" * 80)
print("STEP 23.4 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.5 — SEASONAL MEASURE BEHAVIOUR AUDIT
# ============================================================

import numpy as np

print("=" * 80)
print("STEP 23.5 — ADP SEASONAL MEASURE BEHAVIOUR AUDIT")
print("=" * 80)

# Numeric candidate data only
x = cand[cand["Value_numeric"].notna()].copy()

x["year"] = x["Month_parsed"].dt.year
x["calendar_month"] = x["Month_parsed"].dt.month

# ------------------------------------------------------------
# 1. Distribution of each candidate
# ------------------------------------------------------------
print("\n1. NUMERIC DISTRIBUTIONS")
print("-" * 60)

dist = (
    x.groupby("Measure")["Value_numeric"]
     .agg(["count", "mean", "median", "std", "min", "max"])
)

display(dist)


# ------------------------------------------------------------
# 2. Pivot measures onto same TA-month
# ------------------------------------------------------------
print("\n2. RELATIONSHIP BETWEEN CANDIDATE MEASURES")
print("-" * 60)

wide = (
    x.pivot(
        index=["Area", "Month_parsed"],
        columns="Measure",
        values="Value_numeric"
    )
    .reset_index()
)

print("TA-month rows:", len(wide))

print("\nPearson correlations:")
display(
    wide[candidate_measures].corr(method="pearson")
)

print("\nSpearman correlations:")
display(
    wide[candidate_measures].corr(method="spearman")
)

if (
    "Occupancy rate" in wide.columns and
    "Capacity utilisation rate" in wide.columns
):
    both = wide[
        ["Occupancy rate", "Capacity utilisation rate"]
    ].dropna()

    diff = (
        both["Occupancy rate"] -
        both["Capacity utilisation rate"]
    )

    print("\nOccupancy rate minus Capacity utilisation rate:")
    print("N               :", len(diff))
    print("Exact equal     :", int((diff == 0).sum()))
    print("Mean difference :", diff.mean())
    print("Median difference:", diff.median())
    print("Min difference  :", diff.min())
    print("Max difference  :", diff.max())


# ------------------------------------------------------------
# 3. Year-level behaviour
# ------------------------------------------------------------
print("\n3. YEAR-LEVEL DISTRIBUTIONS")
print("-" * 60)

year_summary = (
    x.groupby(["Measure", "year"])["Value_numeric"]
     .agg(
         n="count",
         mean="mean",
         median="median",
         std="std"
     )
     .reset_index()
)

display(year_summary)


# ------------------------------------------------------------
# 4. Calendar-month pattern by year
# ------------------------------------------------------------
print("\n4. MONTH-OF-YEAR PATTERN BY YEAR")
print("-" * 60)

month_year = (
    x.groupby(
        ["Measure", "year", "calendar_month"]
    )["Value_numeric"]
    .agg(
        n="count",
        median="median",
        mean="mean"
    )
    .reset_index()
)

for measure in candidate_measures:
    print(f"\n--- {measure} ---")

    temp = month_year[
        month_year["Measure"] == measure
    ].pivot(
        index="calendar_month",
        columns="year",
        values="median"
    )

    display(temp)


# ------------------------------------------------------------
# 5. Occupancy: 2022 vs 2023-2024 seasonal pattern correlation
# ------------------------------------------------------------
print("\n5. OCCUPANCY SEASONAL PATTERN STABILITY")
print("-" * 60)

occ = x[x["Measure"] == "Occupancy rate"].copy()

occ_month_year = (
    occ.groupby(["year", "calendar_month"])["Value_numeric"]
       .median()
       .unstack(0)
)

display(occ_month_year)

if all(y in occ_month_year.columns for y in [2022, 2023, 2024]):

    print("\nMonth-pattern correlations:")
    print("2022 vs 2023:",
          occ_month_year[2022].corr(occ_month_year[2023]))

    print("2022 vs 2024:",
          occ_month_year[2022].corr(occ_month_year[2024]))

    print("2023 vs 2024:",
          occ_month_year[2023].corr(occ_month_year[2024]))


print("\n" + "=" * 80)
print("STEP 23.5 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.6 — PRIMARY WINDOW TA-SPECIFIC OCCUPANCY AUDIT
# Candidate primary period: Jan 2023 – Dec 2024
# ============================================================

print("=" * 80)
print("STEP 23.6 — 2023–2024 TA-SPECIFIC OCCUPANCY AUDIT")
print("=" * 80)

occ_2324 = adp[
    (adp["Area type"] == "TA") &
    (adp["Area"] != "Total New Zealand") &
    (adp["Property"] == "Total") &
    (adp["Measure"] == "Occupancy rate") &
    (adp["Month_parsed"].between(
        pd.Timestamp("2023-01-01"),
        pd.Timestamp("2024-12-01")
    ))
].copy()

print("\n1. BASIC COVERAGE")
print("-" * 60)

print("Rows:", len(occ_2324))
print("Expected rows:", 67 * 24)
print("Unique TAs:", occ_2324["Area"].nunique())
print("Unique months:", occ_2324["Month_parsed"].nunique())
print("Numeric observations:", occ_2324["Value_numeric"].notna().sum())
print("Confidentialised/unavailable:",
      occ_2324["Value_numeric"].isna().sum())

# ------------------------------------------------------------
# Coverage by TA
# ------------------------------------------------------------

ta_cov = (
    occ_2324.groupby("Area")
    .agg(
        total_months=("Month_parsed", "size"),
        numeric_months=("Value_numeric", lambda s: s.notna().sum())
    )
    .reset_index()
)

ta_cov["missing_months"] = (
    ta_cov["total_months"] - ta_cov["numeric_months"]
)

print("\n2. TA COVERAGE")
print("-" * 60)

print("Complete 24/24:",
      (ta_cov["numeric_months"] == 24).sum())

print("At least 18/24:",
      (ta_cov["numeric_months"] >= 18).sum())

print("At least 12/24:",
      (ta_cov["numeric_months"] >= 12).sum())

print("Zero numeric:",
      (ta_cov["numeric_months"] == 0).sum())

print("\nIncomplete TAs:")
display(
    ta_cov[
        ta_cov["numeric_months"] < 24
    ].sort_values(
        ["numeric_months", "Area"]
    )
)

# ------------------------------------------------------------
# Critical test:
# observations per calendar month within each TA
# ------------------------------------------------------------

occ_2324["calendar_month"] = occ_2324["Month_parsed"].dt.month

ta_month_cov = (
    occ_2324.groupby(["Area", "calendar_month"])
    .agg(
        available_years=(
            "Value_numeric",
            lambda s: s.notna().sum()
        )
    )
    .reset_index()
)

print("\n3. TA × CALENDAR-MONTH COVERAGE")
print("-" * 60)

print("Expected TA×month-of-year cells:", 67 * 12)

print(
    ta_month_cov["available_years"]
    .value_counts()
    .sort_index()
)

print("\nCells with zero historical observations:",
      (ta_month_cov["available_years"] == 0).sum())

print("Cells with only one historical year:",
      (ta_month_cov["available_years"] == 1).sum())

print("Cells with both 2023 and 2024:",
      (ta_month_cov["available_years"] == 2).sum())

# ------------------------------------------------------------
# Identify problematic TA × month cells
# ------------------------------------------------------------

print("\n4. TA × MONTH CELLS WITHOUT BOTH YEARS")
print("-" * 60)

problem_cells = ta_month_cov[
    ta_month_cov["available_years"] < 2
].copy()

display(problem_cells)

print("\n" + "=" * 80)
print("STEP 23.6 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.7 — NATIONAL OCCUPANCY FALLBACK AUDIT
# Historical period: Jan 2023 – Dec 2024
# ============================================================

print("=" * 80)
print("STEP 23.7 — NATIONAL OCCUPANCY FALLBACK AUDIT")
print("=" * 80)

nz_occ = adp[
    (adp["Area type"] == "TA") &
    (adp["Area"] == "Total New Zealand") &
    (adp["Property"] == "Total") &
    (adp["Measure"] == "Occupancy rate") &
    (adp["Month_parsed"].between(
        pd.Timestamp("2023-01-01"),
        pd.Timestamp("2024-12-01")
    ))
].copy()

print("\n1. NATIONAL SERIES COVERAGE")
print("-" * 60)

print("Rows:", len(nz_occ))
print("Expected:", 24)
print("Numeric:", nz_occ["Value_numeric"].notna().sum())
print("Non-numeric:", nz_occ["Value_numeric"].isna().sum())
print("Unique months:", nz_occ["Month_parsed"].nunique())
print("Earliest:", nz_occ["Month_parsed"].min())
print("Latest:", nz_occ["Month_parsed"].max())

print("\nRaw national series:")
display(
    nz_occ[
        ["Month_parsed", "Value", "Value_numeric", "Flag"]
    ].sort_values("Month_parsed")
)

# ------------------------------------------------------------
# Historical month-of-year means
# ------------------------------------------------------------

nz_occ["calendar_month"] = nz_occ["Month_parsed"].dt.month

nz_month = (
    nz_occ.groupby("calendar_month")
          .agg(
              historical_occ=("Value_numeric", "mean"),
              available_years=("Value_numeric", lambda s: s.notna().sum())
          )
          .reset_index()
)

# Overall national historical baseline
nz_baseline = nz_occ["Value_numeric"].mean()

nz_month["seasonal_index"] = (
    nz_month["historical_occ"] / nz_baseline
)

print("\n2. NATIONAL HISTORICAL BASELINE")
print("-" * 60)

print("Overall 2023–2024 mean occupancy:", nz_baseline)

print("\n3. NATIONAL MONTH-OF-YEAR SEASONAL PROFILE")
print("-" * 60)

display(nz_month)

print("\nSeasonal-index mean:")
print(nz_month["seasonal_index"].mean())

print("\nSeasonal-index range:")
print(
    nz_month["seasonal_index"].min(),
    "to",
    nz_month["seasonal_index"].max()
)

print("\n" + "=" * 80)
print("STEP 23.7 COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.8 — BUILD TA-SPECIFIC HISTORICAL SEASONAL PROFILE
# Primary historical period: Jan 2023 – Dec 2024
# ============================================================

print("=" * 80)
print("STEP 23.8 — BUILD HISTORICAL SEASONAL PROFILE")
print("=" * 80)

# ------------------------------------------------------------
# 1. TA-specific occupancy data
# ------------------------------------------------------------

ta_occ = adp[
    (adp["Area type"] == "TA") &
    (adp["Area"] != "Total New Zealand") &
    (adp["Property"] == "Total") &
    (adp["Measure"] == "Occupancy rate") &
    (adp["Month_parsed"].between(
        pd.Timestamp("2023-01-01"),
        pd.Timestamp("2024-12-01")
    ))
].copy()

ta_occ["calendar_month"] = ta_occ["Month_parsed"].dt.month


# ------------------------------------------------------------
# 2. TA overall historical occupancy baseline
#    Used only when TA has complete 24/24 numeric months
# ------------------------------------------------------------

ta_baseline = (
    ta_occ.groupby("Area")
          .agg(
              ta_numeric_months=(
                  "Value_numeric",
                  lambda s: s.notna().sum()
              ),
              ta_historical_occ_mean=(
                  "Value_numeric",
                  "mean"
              )
          )
          .reset_index()
)

ta_baseline["ta_baseline_eligible"] = (
    ta_baseline["ta_numeric_months"] == 24
)


# ------------------------------------------------------------
# 3. TA × calendar-month historical mean
# ------------------------------------------------------------

ta_month = (
    ta_occ.groupby(["Area", "calendar_month"])
          .agg(
              ta_month_occ_mean=("Value_numeric", "mean"),
              available_years=(
                  "Value_numeric",
                  lambda s: s.notna().sum()
              )
          )
          .reset_index()
)

ta_month = ta_month.merge(
    ta_baseline,
    on="Area",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 4. National fallback profile from Step 23.7
# ------------------------------------------------------------

national_profile = nz_month[
    ["calendar_month", "historical_occ", "seasonal_index"]
].copy()

national_profile = national_profile.rename(
    columns={
        "historical_occ": "nz_month_occ_mean",
        "seasonal_index": "nz_seasonal_index"
    }
)

ta_month = ta_month.merge(
    national_profile,
    on="calendar_month",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 5. TA-specific index eligibility
#
# Conservative rule:
# - TA must have complete 24/24 historical numeric months
# - target calendar month must contain both years
# ------------------------------------------------------------

ta_month["use_ta_specific"] = (
    ta_month["ta_baseline_eligible"] &
    (ta_month["available_years"] == 2)
)


# ------------------------------------------------------------
# 6. Compute TA-specific seasonal index
# ------------------------------------------------------------

ta_month["ta_seasonal_index"] = (
    ta_month["ta_month_occ_mean"] /
    ta_month["ta_historical_occ_mean"]
)


# ------------------------------------------------------------
# 7. Final historical seasonal index
# ------------------------------------------------------------

ta_month["historical_seasonal_index"] = np.where(
    ta_month["use_ta_specific"],
    ta_month["ta_seasonal_index"],
    ta_month["nz_seasonal_index"]
)

ta_month["seasonality_source"] = np.where(
    ta_month["use_ta_specific"],
    "TA_2023_2024",
    "NZ_fallback_2023_2024"
)


# ------------------------------------------------------------
# 8. QC
# ------------------------------------------------------------

print("\n1. FINAL PROFILE DIMENSIONS")
print("-" * 60)

print("Rows:", len(ta_month))
print("Expected:", 67 * 12)
print("Unique TAs:", ta_month["Area"].nunique())
print("Calendar months:", ta_month["calendar_month"].nunique())

print("\n2. SOURCE ASSIGNMENT")
print("-" * 60)

print(ta_month["seasonality_source"].value_counts())

print("\nPercent TA-specific:")
print(
    100 *
    (ta_month["seasonality_source"] == "TA_2023_2024").mean()
)

print("\n3. FINAL INDEX QC")
print("-" * 60)

print(
    ta_month["historical_seasonal_index"]
    .describe()
)

print(
    "\nMissing final seasonal index:",
    ta_month["historical_seasonal_index"].isna().sum()
)

print(
    "Infinite final seasonal index:",
    np.isinf(
        ta_month["historical_seasonal_index"]
    ).sum()
)

print("\n4. FALLBACK TAs / MONTHS")
print("-" * 60)

fallback = ta_month[
    ta_month["seasonality_source"] ==
    "NZ_fallback_2023_2024"
].copy()

print("Fallback rows:", len(fallback))
print("Unique fallback TAs:", fallback["Area"].nunique())

display(
    fallback[
        [
            "Area",
            "calendar_month",
            "ta_numeric_months",
            "available_years",
            "nz_seasonal_index",
            "historical_seasonal_index"
        ]
    ]
)

print("\n" + "=" * 80)
print("STEP 23.8 COMPLETE — NOT YET SAVED")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.8A — AUDIT EXTREME TA SEASONAL INDICES
# ============================================================

print("=" * 80)
print("STEP 23.8A — EXTREME SEASONAL INDEX AUDIT")
print("=" * 80)

# TA-specific rows only
ta_specific = ta_month[
    ta_month["seasonality_source"] == "TA_2023_2024"
].copy()

print("\n1. LOWEST 15 TA-SPECIFIC SEASONAL INDICES")
print("-" * 60)

display(
    ta_specific[
        [
            "Area",
            "calendar_month",
            "ta_month_occ_mean",
            "ta_historical_occ_mean",
            "available_years",
            "ta_seasonal_index"
        ]
    ]
    .sort_values("ta_seasonal_index")
    .head(15)
)


print("\n2. HIGHEST 15 TA-SPECIFIC SEASONAL INDICES")
print("-" * 60)

display(
    ta_specific[
        [
            "Area",
            "calendar_month",
            "ta_month_occ_mean",
            "ta_historical_occ_mean",
            "available_years",
            "ta_seasonal_index"
        ]
    ]
    .sort_values("ta_seasonal_index", ascending=False)
    .head(15)
)


# ------------------------------------------------------------
# Identify exact raw observations behind the extremes
# ------------------------------------------------------------

extreme_cells = pd.concat([
    ta_specific.nsmallest(
        5, "ta_seasonal_index"
    )[["Area", "calendar_month"]],
    
    ta_specific.nlargest(
        5, "ta_seasonal_index"
    )[["Area", "calendar_month"]]
]).drop_duplicates()


extreme_raw = ta_occ.merge(
    extreme_cells,
    on=["Area", "calendar_month"],
    how="inner"
)

print("\n3. RAW 2023–2024 OBSERVATIONS BEHIND TOP/BOTTOM EXTREMES")
print("-" * 60)

display(
    extreme_raw[
        [
            "Area",
            "Month_parsed",
            "calendar_month",
            "Value",
            "Value_numeric",
            "Flag"
        ]
    ]
    .sort_values(
        ["Area", "calendar_month", "Month_parsed"]
    )
)


print("\n4. EXTREME COUNTS")
print("-" * 60)

print(
    "TA-specific index < 0.50:",
    (ta_specific["ta_seasonal_index"] < 0.50).sum()
)

print(
    "TA-specific index > 1.50:",
    (ta_specific["ta_seasonal_index"] > 1.50).sum()
)

print(
    "TA-specific index > 2.00:",
    (ta_specific["ta_seasonal_index"] > 2.00).sum()
)


print("\n" + "=" * 80)
print("STEP 23.8A COMPLETE")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.9 — ADP TA → STATSNZ TA2025 CROSSWALK AUDIT
# ============================================================

import geopandas as gpd
import unicodedata
import re
from pathlib import Path

print("=" * 80)
print("STEP 23.9 — ADP TA → TA2025 CROSSWALK AUDIT")
print("=" * 80)

PROJECT_ROOT = Path(
    r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI"
)

GEO_DIR = (
    PROJECT_ROOT /
    "01_raw" /
    "geography" /
    "statsnz-territorial-authority-2025-SHP"
)

shp_files = list(GEO_DIR.glob("*.shp"))

print("\n1. TA2025 SHAPEFILE")
print("-" * 60)

print("Shapefiles found:", len(shp_files))

for f in shp_files:
    print(f.name)

if len(shp_files) != 1:
    raise ValueError(
        f"Expected exactly 1 shapefile, found {len(shp_files)}"
    )

ta2025 = gpd.read_file(shp_files[0])

print("\nRows:", len(ta2025))
print("CRS:", ta2025.crs)


# ------------------------------------------------------------
# 2. Keep official TA fields
# ------------------------------------------------------------

ta_ref = ta2025[
    [
        "TA2025_V1_",
        "TA2025_V_1",
        "TA2025_V_2"
    ]
].copy()

ta_ref.columns = [
    "ta_code",
    "ta_name_official",
    "ta_name_ascii"
]

ta_ref["ta_code"] = (
    ta_ref["ta_code"]
    .astype(str)
    .str.zfill(3)
)

# Actual 67 TAs only
ta_ref_actual = ta_ref[
    ta_ref["ta_code"] != "999"
].copy()

print("\n2. TA2025 REFERENCE")
print("-" * 60)

print("All TA2025 records:", len(ta_ref))
print("Actual TAs excluding 999:", len(ta_ref_actual))
print("Unique TA codes:",
      ta_ref_actual["ta_code"].nunique())


# ------------------------------------------------------------
# 3. ADP TA names
# ------------------------------------------------------------

adp_tas = (
    ta_month[["Area"]]
    .drop_duplicates()
    .rename(columns={"Area": "adp_ta_name"})
    .copy()
)

print("\n3. ADP TA REFERENCE")
print("-" * 60)

print("Unique ADP actual TAs:", len(adp_tas))


# ------------------------------------------------------------
# 4. Deterministic ASCII normalization
# ------------------------------------------------------------

def normalize_ascii_name(x):
    x = str(x).strip()

    x = unicodedata.normalize("NFKD", x)
    x = "".join(
        c for c in x
        if not unicodedata.combining(c)
    )

    x = x.casefold()

    # normalize punctuation/spacing only
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x


adp_tas["match_name"] = (
    adp_tas["adp_ta_name"]
    .map(normalize_ascii_name)
)

ta_ref_actual["match_name"] = (
    ta_ref_actual["ta_name_ascii"]
    .map(normalize_ascii_name)
)


# ------------------------------------------------------------
# 5. Exact deterministic crosswalk
# ------------------------------------------------------------

crosswalk = adp_tas.merge(
    ta_ref_actual,
    on="match_name",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# 6. QC
# ------------------------------------------------------------

print("\n4. CROSSWALK QC")
print("-" * 60)

print("Crosswalk rows:", len(crosswalk))

print(
    "Matched TA codes:",
    crosswalk["ta_code"].notna().sum()
)

print(
    "Unmatched:",
    crosswalk["ta_code"].isna().sum()
)

print(
    "Unique matched TA codes:",
    crosswalk["ta_code"].nunique()
)

print(
    "Duplicate ADP names:",
    crosswalk["adp_ta_name"].duplicated().sum()
)

print(
    "Duplicate matched TA codes:",
    crosswalk["ta_code"].dropna().duplicated().sum()
)


print("\n5. UNMATCHED RECORDS")
print("-" * 60)

display(
    crosswalk[
        crosswalk["ta_code"].isna()
    ][
        [
            "adp_ta_name",
            "match_name"
        ]
    ]
)


print("\n6. NAME DIFFERENCES AFTER SUCCESSFUL MATCH")
print("-" * 60)

name_diff = crosswalk[
    crosswalk["adp_ta_name"] !=
    crosswalk["ta_name_official"]
][
    [
        "adp_ta_name",
        "ta_code",
        "ta_name_official",
        "ta_name_ascii"
    ]
]

display(name_diff)


print("\n7. FULL CROSSWALK")
print("-" * 60)

display(
    crosswalk[
        [
            "adp_ta_name",
            "ta_code",
            "ta_name_official",
            "ta_name_ascii"
        ]
    ].sort_values("ta_code")
)


print("\n" + "=" * 80)

if (
    len(crosswalk) == 67
    and crosswalk["ta_code"].notna().sum() == 67
    and crosswalk["ta_code"].nunique() == 67
):
    print("STEP 23.9 STATUS: PASS")
else:
    print("STEP 23.9 STATUS: REVIEW REQUIRED")

print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.10 — FREEZE HISTORICAL ADP SEASONALITY PROFILE
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 23.10 — FREEZE HISTORICAL ADP SEASONALITY PROFILE")
print("=" * 80)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "02_intermediate" /
    "seasonality"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "adp_historical_seasonality_ta_month_frozen.parquet"
)


# ------------------------------------------------------------
# 1. Attach TA2025 codes
# ------------------------------------------------------------

seasonality_frozen = ta_month.merge(
    crosswalk[
        [
            "adp_ta_name",
            "ta_code",
            "ta_name_official",
            "ta_name_ascii"
        ]
    ],
    left_on="Area",
    right_on="adp_ta_name",
    how="left",
    validate="many_to_one"
)


# ------------------------------------------------------------
# 2. Keep only frozen analytical fields
# ------------------------------------------------------------

seasonality_frozen = seasonality_frozen[
    [
        "ta_code",
        "ta_name_official",
        "ta_name_ascii",
        "calendar_month",
        "historical_seasonal_index",
        "seasonality_source"
    ]
].copy()


# ------------------------------------------------------------
# 3. Standardise data types
# ------------------------------------------------------------

seasonality_frozen["ta_code"] = (
    seasonality_frozen["ta_code"]
    .astype(str)
    .str.zfill(3)
)

seasonality_frozen["calendar_month"] = (
    seasonality_frozen["calendar_month"]
    .astype(int)
)

seasonality_frozen["historical_seasonal_index"] = (
    seasonality_frozen["historical_seasonal_index"]
    .astype(float)
)


# ------------------------------------------------------------
# 4. Sort deterministically
# ------------------------------------------------------------

seasonality_frozen = (
    seasonality_frozen
    .sort_values(
        ["ta_code", "calendar_month"]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Pre-save QC
# ------------------------------------------------------------

print("\n1. PRE-SAVE QC")
print("-" * 60)

print("Rows:", len(seasonality_frozen))
print("Expected:", 67 * 12)

print(
    "Unique TA codes:",
    seasonality_frozen["ta_code"].nunique()
)

print(
    "Unique calendar months:",
    seasonality_frozen["calendar_month"].nunique()
)

print(
    "Duplicate TA × month keys:",
    seasonality_frozen.duplicated(
        ["ta_code", "calendar_month"]
    ).sum()
)

print(
    "Missing TA codes:",
    seasonality_frozen["ta_code"].isna().sum()
)

print(
    "Missing seasonal index:",
    seasonality_frozen[
        "historical_seasonal_index"
    ].isna().sum()
)

print(
    "Infinite seasonal index:",
    np.isinf(
        seasonality_frozen[
            "historical_seasonal_index"
        ]
    ).sum()
)

print("\nSource distribution:")
print(
    seasonality_frozen[
        "seasonality_source"
    ].value_counts()
)

print("\nIndex distribution:")
print(
    seasonality_frozen[
        "historical_seasonal_index"
    ].describe()
)


# ------------------------------------------------------------
# 6. Hard QC before saving
# ------------------------------------------------------------

assert len(seasonality_frozen) == 804
assert seasonality_frozen["ta_code"].nunique() == 67
assert seasonality_frozen["calendar_month"].nunique() == 12

assert (
    seasonality_frozen.duplicated(
        ["ta_code", "calendar_month"]
    ).sum()
    == 0
)

assert (
    seasonality_frozen["historical_seasonal_index"]
    .isna()
    .sum()
    == 0
)

assert (
    np.isinf(
        seasonality_frozen[
            "historical_seasonal_index"
        ]
    ).sum()
    == 0
)

assert (
    seasonality_frozen["seasonality_source"]
    .value_counts()
    .get("TA_2023_2024", 0)
    == 696
)

assert (
    seasonality_frozen["seasonality_source"]
    .value_counts()
    .get("NZ_fallback_2023_2024", 0)
    == 108
)


# ------------------------------------------------------------
# 7. Save
# ------------------------------------------------------------

seasonality_frozen.to_parquet(
    OUTPUT_FILE,
    index=False
)

print("\n2. FILE SAVED")
print("-" * 60)
print(OUTPUT_FILE)


# ------------------------------------------------------------
# 8. Reload and exact integrity check
# ------------------------------------------------------------

reload = pd.read_parquet(
    OUTPUT_FILE
)

print("\n3. RELOAD QC")
print("-" * 60)

print("Reload rows:", len(reload))
print("Reload columns:", len(reload.columns))

print(
    "Same shape:",
    reload.shape == seasonality_frozen.shape
)

print(
    "Same columns:",
    list(reload.columns) ==
    list(seasonality_frozen.columns)
)

print(
    "Same TA × month keys:",
    reload[
        ["ta_code", "calendar_month"]
    ].equals(
        seasonality_frozen[
            ["ta_code", "calendar_month"]
        ]
    )
)

print(
    "Seasonal index exact:",
    np.allclose(
        reload["historical_seasonal_index"],
        seasonality_frozen[
            "historical_seasonal_index"
        ],
        equal_nan=True
    )
)

print(
    "Source exact:",
    reload["seasonality_source"].equals(
        seasonality_frozen[
            "seasonality_source"
        ]
    )
)


# ------------------------------------------------------------
# 9. Final preview
# ------------------------------------------------------------

print("\n4. FROZEN TABLE PREVIEW")
print("-" * 60)

display(reload.head(15))


print("\n" + "=" * 80)

if (
    reload.shape == seasonality_frozen.shape
    and len(reload) == 804
    and reload["ta_code"].nunique() == 67
    and reload.duplicated(
        ["ta_code", "calendar_month"]
    ).sum() == 0
    and reload[
        "historical_seasonal_index"
    ].isna().sum() == 0
):
    print(
        "STEP 23.10 STATUS: "
        "HISTORICAL SEASONALITY PROFILE FROZEN"
    )
else:
    print(
        "STEP 23.10 STATUS: REVIEW REQUIRED"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.11 — FORWARD-INTERVAL SEASONAL WEIGHT AUDIT
# ============================================================

import pandas as pd

print("=" * 80)
print("STEP 23.11 — FORWARD-INTERVAL SEASONAL WEIGHT AUDIT")
print("=" * 80)

transitions = pd.DataFrame({
    "transition": [
        "T1", "T2", "T3", "T4", "T5",
        "T6", "T7", "T8", "T9", "T10"
    ],
    "anchor_date": pd.to_datetime([
        "2025-10-05",
        "2025-11-07",
        "2025-12-11",
        "2026-01-16",
        "2026-02-13",
        "2026-03-17",
        "2026-04-16",
        "2026-05-23",
        "2026-06-19",
        "2026-07-12"
    ]),
    "future_date": pd.to_datetime([
        "2025-11-07",
        "2025-12-11",
        "2026-01-16",
        "2026-02-13",
        "2026-03-17",
        "2026-04-16",
        "2026-05-23",
        "2026-06-19",
        "2026-07-12",
        "2026-08-13"
    ])
})

# ------------------------------------------------------------
# Convention:
# include anchor date
# exclude future snapshot date
#
# [anchor_date, future_date)
# ------------------------------------------------------------

records = []

for _, r in transitions.iterrows():

    days = pd.date_range(
        start=r["anchor_date"],
        end=r["future_date"] - pd.Timedelta(days=1),
        freq="D"
    )

    horizon_days = len(days)

    counts = (
        pd.Series(days.month)
        .value_counts()
        .sort_index()
    )

    for month, n_days in counts.items():

        records.append({
            "transition": r["transition"],
            "anchor_date": r["anchor_date"],
            "future_date": r["future_date"],
            "forecast_horizon_days": horizon_days,
            "calendar_month": int(month),
            "days_in_interval": int(n_days),
            "weight": n_days / horizon_days
        })

weights = pd.DataFrame(records)


# ------------------------------------------------------------
# 1. Transition summary
# ------------------------------------------------------------

print("\n1. FORECAST HORIZONS")
print("-" * 60)

display(
    transitions.assign(
        forecast_horizon_days=(
            transitions["future_date"] -
            transitions["anchor_date"]
        ).dt.days
    )
)


# ------------------------------------------------------------
# 2. Month weights
# ------------------------------------------------------------

print("\n2. CALENDAR-MONTH WEIGHTS")
print("-" * 60)

display(weights)


# ------------------------------------------------------------
# 3. QC
# ------------------------------------------------------------

print("\n3. WEIGHT QC")
print("-" * 60)

qc = (
    weights.groupby("transition")
    .agg(
        horizon_days=("forecast_horizon_days", "first"),
        allocated_days=("days_in_interval", "sum"),
        weight_sum=("weight", "sum"),
        months_spanned=("calendar_month", "nunique")
    )
    .reset_index()
)

display(qc)

print(
    "\nTransitions with allocated-day mismatch:",
    (qc["horizon_days"] != qc["allocated_days"]).sum()
)

print(
    "Transitions with weight sum != 1:",
    (~qc["weight_sum"].round(12).eq(1.0)).sum()
)

print(
    "Minimum horizon:",
    qc["horizon_days"].min()
)

print(
    "Maximum horizon:",
    qc["horizon_days"].max()
)


print("\n" + "=" * 80)
print("STEP 23.11 COMPLETE — NO FEATURE CREATED YET")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.12 — BUILD TA × TRANSITION SEASONALITY FEATURES
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 23.12 — BUILD TA × TRANSITION SEASONALITY FEATURES")
print("=" * 80)


# ------------------------------------------------------------
# 1. Merge month weights with frozen historical profile
# ------------------------------------------------------------

seasonality_weighted = weights.merge(
    seasonality_frozen[
        [
            "ta_code",
            "ta_name_official",
            "calendar_month",
            "historical_seasonal_index",
            "seasonality_source"
        ]
    ],
    on="calendar_month",
    how="left",
    validate="many_to_many"
)


print("\n1. MONTH-LEVEL EXPANDED TABLE")
print("-" * 60)

print("Rows:", len(seasonality_weighted))
print("Expected:", len(weights) * 67)

print(
    "Missing historical seasonal index:",
    seasonality_weighted[
        "historical_seasonal_index"
    ].isna().sum()
)


# ------------------------------------------------------------
# 2. Weighted contribution
# ------------------------------------------------------------

seasonality_weighted[
    "weighted_seasonality"
] = (
    seasonality_weighted["weight"] *
    seasonality_weighted["historical_seasonal_index"]
)


# ------------------------------------------------------------
# 3. Aggregate to TA × transition
# ------------------------------------------------------------

seasonality_transition = (
    seasonality_weighted
    .groupby(
        [
            "transition",
            "ta_code",
            "ta_name_official"
        ],
        as_index=False
    )
    .agg(
        anchor_date=("anchor_date", "first"),
        future_date=("future_date", "first"),
        forecast_horizon_days=(
            "forecast_horizon_days",
            "first"
        ),
        historical_seasonality_index_horizon=(
            "weighted_seasonality",
            "sum"
        ),
        months_spanned=(
            "calendar_month",
            "nunique"
        )
    )
)


# ------------------------------------------------------------
# 4. Determine provenance of each TA profile
# ------------------------------------------------------------

source_by_ta = (
    seasonality_frozen
    .groupby("ta_code")["seasonality_source"]
    .agg(lambda x: x.iloc[0])
    .reset_index()
)

seasonality_transition = (
    seasonality_transition.merge(
        source_by_ta,
        on="ta_code",
        how="left",
        validate="many_to_one"
    )
)


# ------------------------------------------------------------
# 5. Deterministic transition order
# ------------------------------------------------------------

transition_order = {
    f"T{i}": i
    for i in range(1, 11)
}

seasonality_transition[
    "_transition_order"
] = (
    seasonality_transition["transition"]
    .map(transition_order)
)

seasonality_transition = (
    seasonality_transition
    .sort_values(
        ["_transition_order", "ta_code"]
    )
    .drop(columns="_transition_order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Structural QC
# ------------------------------------------------------------

print("\n2. FINAL TABLE STRUCTURE")
print("-" * 60)

print("Rows:", len(seasonality_transition))
print("Expected:", 67 * 10)

print(
    "Unique TAs:",
    seasonality_transition["ta_code"].nunique()
)

print(
    "Unique transitions:",
    seasonality_transition[
        "transition"
    ].nunique()
)

print(
    "Duplicate TA × transition keys:",
    seasonality_transition.duplicated(
        ["transition", "ta_code"]
    ).sum()
)

print(
    "Missing weighted index:",
    seasonality_transition[
        "historical_seasonality_index_horizon"
    ].isna().sum()
)

print(
    "Infinite weighted index:",
    np.isinf(
        seasonality_transition[
            "historical_seasonality_index_horizon"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 7. Coverage per transition
# ------------------------------------------------------------

print("\n3. COVERAGE BY TRANSITION")
print("-" * 60)

coverage = (
    seasonality_transition
    .groupby("transition")
    .agg(
        rows=("ta_code", "size"),
        unique_tas=("ta_code", "nunique"),
        horizon_days=(
            "forecast_horizon_days",
            "first"
        ),
        min_index=(
            "historical_seasonality_index_horizon",
            "min"
        ),
        mean_index=(
            "historical_seasonality_index_horizon",
            "mean"
        ),
        max_index=(
            "historical_seasonality_index_horizon",
            "max"
        )
    )
    .reset_index()
)

coverage["_order"] = coverage["transition"].map(
    transition_order
)

coverage = (
    coverage
    .sort_values("_order")
    .drop(columns="_order")
)

display(coverage)


# ------------------------------------------------------------
# 8. Source QC
# ------------------------------------------------------------

print("\n4. SEASONALITY SOURCE BY TA × TRANSITION")
print("-" * 60)

print(
    seasonality_transition[
        "seasonality_source"
    ].value_counts()
)


# ------------------------------------------------------------
# 9. Overall distribution
# ------------------------------------------------------------

print("\n5. WEIGHTED SEASONALITY DISTRIBUTION")
print("-" * 60)

print(
    seasonality_transition[
        "historical_seasonality_index_horizon"
    ].describe()
)


# ------------------------------------------------------------
# 10. Preview selected TAs
# ------------------------------------------------------------

print("\n6. EXAMPLE — FAR NORTH")
print("-" * 60)

display(
    seasonality_transition[
        seasonality_transition["ta_code"] == "001"
    ]
)


print("\n7. EXAMPLE — ONE NATIONAL-FALLBACK TA")
print("-" * 60)

fallback_codes = (
    seasonality_transition.loc[
        seasonality_transition[
            "seasonality_source"
        ] == "NZ_fallback_2023_2024",
        "ta_code"
    ]
    .drop_duplicates()
    .tolist()
)

if fallback_codes:
    example_fallback = fallback_codes[0]

    display(
        seasonality_transition[
            seasonality_transition[
                "ta_code"
            ] == example_fallback
        ]
    )


# ------------------------------------------------------------
# 11. Hard checks
# ------------------------------------------------------------

assert len(seasonality_transition) == 670

assert (
    seasonality_transition[
        "ta_code"
    ].nunique() == 67
)

assert (
    seasonality_transition[
        "transition"
    ].nunique() == 10
)

assert (
    seasonality_transition.duplicated(
        ["transition", "ta_code"]
    ).sum() == 0
)

assert (
    seasonality_transition[
        "historical_seasonality_index_horizon"
    ].isna().sum() == 0
)

assert (
    np.isinf(
        seasonality_transition[
            "historical_seasonality_index_horizon"
        ]
    ).sum() == 0
)


print("\n" + "=" * 80)
print("STEP 23.12 COMPLETE — NOT YET SAVED")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.13 — FREEZE TA × TRANSITION SEASONALITY TABLE
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 23.13 — FREEZE TA × TRANSITION SEASONALITY TABLE")
print("=" * 80)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "02_intermediate" /
    "seasonality"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "seasonality_features_ta_transition_frozen.parquet"
)


# ------------------------------------------------------------
# 1. Select frozen output fields
# ------------------------------------------------------------

seasonality_final = seasonality_transition[
    [
        "transition",
        "ta_code",
        "anchor_date",
        "future_date",
        "forecast_horizon_days",
        "historical_seasonality_index_horizon",
        "seasonality_source"
    ]
].copy()


# ------------------------------------------------------------
# 2. Standardise types
# ------------------------------------------------------------

seasonality_final["ta_code"] = (
    seasonality_final["ta_code"]
    .astype(str)
    .str.zfill(3)
)

seasonality_final["forecast_horizon_days"] = (
    seasonality_final["forecast_horizon_days"]
    .astype(int)
)

seasonality_final[
    "historical_seasonality_index_horizon"
] = (
    seasonality_final[
        "historical_seasonality_index_horizon"
    ].astype(float)
)


# ------------------------------------------------------------
# 3. Deterministic ordering
# ------------------------------------------------------------

transition_order = {
    f"T{i}": i
    for i in range(1, 11)
}

seasonality_final["_order"] = (
    seasonality_final["transition"]
    .map(transition_order)
)

seasonality_final = (
    seasonality_final
    .sort_values(
        ["_order", "ta_code"]
    )
    .drop(columns="_order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Pre-save QC
# ------------------------------------------------------------

print("\n1. PRE-SAVE QC")
print("-" * 60)

print("Rows:", len(seasonality_final))
print("Expected:", 67 * 10)

print(
    "Unique transitions:",
    seasonality_final["transition"].nunique()
)

print(
    "Unique TA codes:",
    seasonality_final["ta_code"].nunique()
)

print(
    "Duplicate transition × TA keys:",
    seasonality_final.duplicated(
        ["transition", "ta_code"]
    ).sum()
)

print(
    "Missing seasonal index:",
    seasonality_final[
        "historical_seasonality_index_horizon"
    ].isna().sum()
)

print(
    "Infinite seasonal index:",
    np.isinf(
        seasonality_final[
            "historical_seasonality_index_horizon"
        ]
    ).sum()
)

print("\nSource distribution:")
print(
    seasonality_final[
        "seasonality_source"
    ].value_counts()
)

print("\nForecast horizons:")
display(
    seasonality_final[
        [
            "transition",
            "anchor_date",
            "future_date",
            "forecast_horizon_days"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Hard checks
# ------------------------------------------------------------

assert len(seasonality_final) == 670

assert (
    seasonality_final["transition"].nunique()
    == 10
)

assert (
    seasonality_final["ta_code"].nunique()
    == 67
)

assert (
    seasonality_final.duplicated(
        ["transition", "ta_code"]
    ).sum()
    == 0
)

assert (
    seasonality_final[
        "historical_seasonality_index_horizon"
    ].isna().sum()
    == 0
)

assert (
    np.isinf(
        seasonality_final[
            "historical_seasonality_index_horizon"
        ]
    ).sum()
    == 0
)

assert (
    seasonality_final[
        "seasonality_source"
    ].value_counts().get(
        "TA_2023_2024", 0
    )
    == 580
)

assert (
    seasonality_final[
        "seasonality_source"
    ].value_counts().get(
        "NZ_fallback_2023_2024", 0
    )
    == 90
)


# ------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------

seasonality_final.to_parquet(
    OUTPUT_FILE,
    index=False
)

print("\n2. SAVED")
print("-" * 60)
print(OUTPUT_FILE)


# ------------------------------------------------------------
# 7. Reload integrity QC
# ------------------------------------------------------------

reload = pd.read_parquet(
    OUTPUT_FILE
)

print("\n3. RELOAD QC")
print("-" * 60)

print("Reload rows:", len(reload))
print("Reload columns:", len(reload.columns))

print(
    "Same shape:",
    reload.shape == seasonality_final.shape
)

print(
    "Same columns:",
    list(reload.columns) ==
    list(seasonality_final.columns)
)

print(
    "Same keys:",
    reload[
        ["transition", "ta_code"]
    ].equals(
        seasonality_final[
            ["transition", "ta_code"]
        ]
    )
)

print(
    "Seasonality exact:",
    np.allclose(
        reload[
            "historical_seasonality_index_horizon"
        ],
        seasonality_final[
            "historical_seasonality_index_horizon"
        ],
        equal_nan=True
    )
)

print(
    "Horizon exact:",
    reload[
        "forecast_horizon_days"
    ].equals(
        seasonality_final[
            "forecast_horizon_days"
        ]
    )
)

print(
    "Source exact:",
    reload[
        "seasonality_source"
    ].equals(
        seasonality_final[
            "seasonality_source"
        ]
    )
)


# ------------------------------------------------------------
# 8. Final status
# ------------------------------------------------------------

print("\n" + "=" * 80)

if (
    reload.shape == seasonality_final.shape
    and len(reload) == 670
    and reload["transition"].nunique() == 10
    and reload["ta_code"].nunique() == 67
    and reload.duplicated(
        ["transition", "ta_code"]
    ).sum() == 0
):
    print(
        "STEP 23.13 STATUS: "
        "SEASONALITY TRANSITION TABLE FROZEN"
    )
else:
    print(
        "STEP 23.13 STATUS: REVIEW REQUIRED"
    )

print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.14 — TA999 NATIONAL SEASONALITY FALLBACK AUDIT
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 23.14 — TA999 NATIONAL SEASONALITY FALLBACK AUDIT")
print("=" * 80)

# National month profile already created in Step 23.7:
# nz_month:
# calendar_month | historical_occ | available_years | seasonal_index

nz_horizon = weights.merge(
    nz_month[
        [
            "calendar_month",
            "seasonal_index"
        ]
    ],
    on="calendar_month",
    how="left",
    validate="many_to_one"
)

nz_horizon["weighted_index"] = (
    nz_horizon["weight"] *
    nz_horizon["seasonal_index"]
)

ta999_fallback = (
    nz_horizon
    .groupby("transition", as_index=False)
    .agg(
        anchor_date=("anchor_date", "first"),
        future_date=("future_date", "first"),
        forecast_horizon_days=(
            "forecast_horizon_days", "first"
        ),
        historical_seasonality_index_horizon=(
            "weighted_index", "sum"
        ),
        months_spanned=("calendar_month", "nunique")
    )
)

# deterministic order
transition_order = {
    f"T{i}": i for i in range(1, 11)
}

ta999_fallback["_order"] = (
    ta999_fallback["transition"]
    .map(transition_order)
)

ta999_fallback = (
    ta999_fallback
    .sort_values("_order")
    .drop(columns="_order")
    .reset_index(drop=True)
)

ta999_fallback["ta_code"] = "999"
ta999_fallback[
    "seasonality_source"
] = "NZ_fallback_TA999_2023_2024"


print("\n1. TA999 FALLBACK TABLE")
print("-" * 60)

display(
    ta999_fallback[
        [
            "transition",
            "ta_code",
            "anchor_date",
            "future_date",
            "forecast_horizon_days",
            "historical_seasonality_index_horizon",
            "seasonality_source"
        ]
    ]
)


print("\n2. QC")
print("-" * 60)

print("Rows:", len(ta999_fallback))
print("Expected:", 10)

print(
    "Unique transitions:",
    ta999_fallback["transition"].nunique()
)

print(
    "Missing index:",
    ta999_fallback[
        "historical_seasonality_index_horizon"
    ].isna().sum()
)

print(
    "Infinite index:",
    np.isinf(
        ta999_fallback[
            "historical_seasonality_index_horizon"
        ]
    ).sum()
)

print(
    "Minimum index:",
    ta999_fallback[
        "historical_seasonality_index_horizon"
    ].min()
)

print(
    "Maximum index:",
    ta999_fallback[
        "historical_seasonality_index_horizon"
    ].max()
)


assert len(ta999_fallback) == 10

assert (
    ta999_fallback["transition"].nunique()
    == 10
)

assert (
    ta999_fallback[
        "historical_seasonality_index_horizon"
    ].isna().sum()
    == 0
)

assert (
    np.isinf(
        ta999_fallback[
            "historical_seasonality_index_horizon"
        ]
    ).sum()
    == 0
)


print("\n" + "=" * 80)
print("STEP 23.14 COMPLETE — NOT YET SAVED")
print("=" * 80)

In [ ]:
# ============================================================
# STEP 23.15 — FREEZE MERGE-READY SEASONALITY TABLE
# 68 TA categories × 10 transitions = 680 rows
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 80)
print("STEP 23.15 — FREEZE MERGE-READY SEASONALITY TABLE")
print("=" * 80)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "02_intermediate" /
    "seasonality"
)

OUTPUT_FILE = (
    OUTPUT_DIR /
    "seasonality_features_ta_transition_merge_ready_frozen.parquet"
)


# ------------------------------------------------------------
# 1. Prepare 67 official TA rows
# ------------------------------------------------------------

official = seasonality_final.copy()


# ------------------------------------------------------------
# 2. Prepare TA999 rows
# ------------------------------------------------------------

ta999 = ta999_fallback[
    [
        "transition",
        "ta_code",
        "anchor_date",
        "future_date",
        "forecast_horizon_days",
        "historical_seasonality_index_horizon",
        "seasonality_source"
    ]
].copy()


# ------------------------------------------------------------
# 3. Combine
# ------------------------------------------------------------

seasonality_merge_ready = pd.concat(
    [official, ta999],
    ignore_index=True
)


# ------------------------------------------------------------
# 4. Standardise types
# ------------------------------------------------------------

seasonality_merge_ready["ta_code"] = (
    seasonality_merge_ready["ta_code"]
    .astype(str)
    .str.zfill(3)
)

seasonality_merge_ready[
    "forecast_horizon_days"
] = (
    seasonality_merge_ready[
        "forecast_horizon_days"
    ].astype(int)
)

seasonality_merge_ready[
    "historical_seasonality_index_horizon"
] = (
    seasonality_merge_ready[
        "historical_seasonality_index_horizon"
    ].astype(float)
)


# ------------------------------------------------------------
# 5. Deterministic order
# ------------------------------------------------------------

transition_order = {
    f"T{i}": i
    for i in range(1, 11)
}

seasonality_merge_ready["_order"] = (
    seasonality_merge_ready["transition"]
    .map(transition_order)
)

seasonality_merge_ready = (
    seasonality_merge_ready
    .sort_values(
        ["_order", "ta_code"]
    )
    .drop(columns="_order")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. PRE-SAVE QC
# ------------------------------------------------------------

print("\n1. STRUCTURAL QC")
print("-" * 60)

print(
    "Rows:",
    len(seasonality_merge_ready)
)

print(
    "Expected:",
    68 * 10
)

print(
    "Unique transitions:",
    seasonality_merge_ready[
        "transition"
    ].nunique()
)

print(
    "Unique TA categories:",
    seasonality_merge_ready[
        "ta_code"
    ].nunique()
)

print(
    "Duplicate transition × TA:",
    seasonality_merge_ready.duplicated(
        ["transition", "ta_code"]
    ).sum()
)

print(
    "Missing seasonal index:",
    seasonality_merge_ready[
        "historical_seasonality_index_horizon"
    ].isna().sum()
)

print(
    "Infinite seasonal index:",
    np.isinf(
        seasonality_merge_ready[
            "historical_seasonality_index_horizon"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 7. Coverage per transition
# ------------------------------------------------------------

print("\n2. COVERAGE BY TRANSITION")
print("-" * 60)

coverage_final = (
    seasonality_merge_ready
    .groupby("transition")
    .agg(
        rows=("ta_code", "size"),
        unique_tas=("ta_code", "nunique"),
        horizon_days=(
            "forecast_horizon_days",
            "first"
        )
    )
    .reset_index()
)

coverage_final["_order"] = (
    coverage_final["transition"]
    .map(transition_order)
)

coverage_final = (
    coverage_final
    .sort_values("_order")
    .drop(columns="_order")
)

display(coverage_final)


# ------------------------------------------------------------
# 8. Source distribution
# ------------------------------------------------------------

print("\n3. SOURCE DISTRIBUTION")
print("-" * 60)

print(
    seasonality_merge_ready[
        "seasonality_source"
    ].value_counts()
)


# ------------------------------------------------------------
# 9. TA999 audit
# ------------------------------------------------------------

print("\n4. TA999 AUDIT")
print("-" * 60)

display(
    seasonality_merge_ready[
        seasonality_merge_ready[
            "ta_code"
        ] == "999"
    ]
)


# ------------------------------------------------------------
# 10. Hard checks
# ------------------------------------------------------------

assert len(seasonality_merge_ready) == 680

assert (
    seasonality_merge_ready[
        "transition"
    ].nunique()
    == 10
)

assert (
    seasonality_merge_ready[
        "ta_code"
    ].nunique()
    == 68
)

assert (
    seasonality_merge_ready.duplicated(
        ["transition", "ta_code"]
    ).sum()
    == 0
)

assert (
    seasonality_merge_ready[
        "historical_seasonality_index_horizon"
    ].isna().sum()
    == 0
)

assert (
    np.isinf(
        seasonality_merge_ready[
            "historical_seasonality_index_horizon"
        ]
    ).sum()
    == 0
)

assert (
    (
        seasonality_merge_ready[
            "ta_code"
        ] == "999"
    ).sum()
    == 10
)

assert (
    seasonality_merge_ready[
        "seasonality_source"
    ].value_counts().get(
        "TA_2023_2024", 0
    )
    == 580
)

assert (
    seasonality_merge_ready[
        "seasonality_source"
    ].value_counts().get(
        "NZ_fallback_2023_2024", 0
    )
    == 90
)

assert (
    seasonality_merge_ready[
        "seasonality_source"
    ].value_counts().get(
        "NZ_fallback_TA999_2023_2024", 0
    )
    == 10
)


# ------------------------------------------------------------
# 11. Save
# ------------------------------------------------------------

seasonality_merge_ready.to_parquet(
    OUTPUT_FILE,
    index=False
)

print("\n5. SAVED")
print("-" * 60)
print(OUTPUT_FILE)


# ------------------------------------------------------------
# 12. Reload integrity QC
# ------------------------------------------------------------

reload = pd.read_parquet(
    OUTPUT_FILE
)

print("\n6. RELOAD QC")
print("-" * 60)

print("Reload rows:", len(reload))
print("Reload columns:", len(reload.columns))

print(
    "Same shape:",
    reload.shape ==
    seasonality_merge_ready.shape
)

print(
    "Same columns:",
    list(reload.columns) ==
    list(seasonality_merge_ready.columns)
)

print(
    "Same keys:",
    reload[
        ["transition", "ta_code"]
    ].equals(
        seasonality_merge_ready[
            ["transition", "ta_code"]
        ]
    )
)

print(
    "Seasonality exact:",
    np.allclose(
        reload[
            "historical_seasonality_index_horizon"
        ],
        seasonality_merge_ready[
            "historical_seasonality_index_horizon"
        ],
        equal_nan=True
    )
)

print(
    "Horizon exact:",
    reload[
        "forecast_horizon_days"
    ].equals(
        seasonality_merge_ready[
            "forecast_horizon_days"
        ]
    )
)

print(
    "Source exact:",
    reload[
        "seasonality_source"
    ].equals(
        seasonality_merge_ready[
            "seasonality_source"
        ]
    )
)


print("\n" + "=" * 80)

if (
    len(reload) == 680
    and reload["transition"].nunique() == 10
    and reload["ta_code"].nunique() == 68
    and reload.duplicated(
        ["transition", "ta_code"]
    ).sum() == 0
    and reload[
        "historical_seasonality_index_horizon"
    ].isna().sum() == 0
):
    print(
        "STEP 23.15 STATUS: "
        "MERGE-READY SEASONALITY TABLE FROZEN"
    )
else:
    print(
        "STEP 23.15 STATUS: REVIEW REQUIRED"
    )

print("=" * 80)